# Exploration of international match results

## 1. Data
I load the raw tables from DuckDB and check their content.

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.database import query

query("SELECT * FROM raw_results ORDER BY date DESC LIMIT 5")

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,2026-08-26,Vietnam,Thailand,2,2,ASEAN Championship,Hanoi,Vietnam,False
1,2026-08-22,Thailand,Vietnam,0,2,ASEAN Championship,Bangkok,Thailand,False
2,2026-08-19,Vietnam,Malaysia,2,0,ASEAN Championship,Hanoi,Vietnam,False
3,2026-08-18,Thailand,Singapore,1,2,ASEAN Championship,Bangkok,Thailand,False
4,2026-08-16,Malaysia,Vietnam,0,2,ASEAN Championship,Kuala Lumpur,Malaysia,False


In [2]:
query("""
    SELECT
        COUNT(*) AS n_matches,
        MIN(date) AS first_match,
        MAX(date) AS last_match,
        COUNT(DISTINCT home_team) AS n_home_teams,
        ROUND(AVG(neutral::INTEGER), 3) AS share_neutral
    FROM raw_results
""")

,n_matches,first_match,last_match,n_home_teams,share_neutral
0,49547,1872-11-30,2026-08-26,328,0.266


The columns are in the right order and the last match is on 2026-08-26, matching the pinned dataset commit.
The data has 328 distinct home teams, more than the 211 FIFA members, so some non-FIFA teams need a decision.
About 27% of matches are played on neutral ground, so home advantage must depend on the `neutral` flag.

## 2. Coverage over time and competitions

In [ ]:
by_decade = query("""
    SELECT
        (YEAR(date) // 10) * 10 AS decade,
        COUNT(*) AS n_matches,
        COUNT(DISTINCT home_team) AS n_teams
    FROM raw_results
    GROUP BY decade
    ORDER BY decade
""")
by_decade

,decade,n_matches,n_teams
0,1870,13,3
1,1880,55,5
2,1890,59,4
3,1900,138,19
4,1910,330,33
5,1920,831,68
6,1930,1079,76
7,1940,833,95
8,1950,1651,124
9,1960,2971,152


In [ ]:
query("""
    SELECT tournament, COUNT(*) AS n_matches
    FROM raw_results
    WHERE date >= '2000-01-01'
    GROUP BY tournament
    ORDER BY n_matches DESC
    LIMIT 20
""")

,tournament,n_matches
0,Friendly,8445
1,FIFA World Cup qualification,5840
2,UEFA Euro qualification,1532
3,African Cup of Nations qualification,1416
4,UEFA Nations League,658
5,AFC Asian Cup qualification,530
6,African Cup of Nations,525
7,FIFA World Cup,488
8,CONCACAF Nations League,422
9,Gold Cup,359


Before 1950 the data is sparse. The volume grows fast from the 1990s.
Friendlies are a third of matches since 2000, and they carry less information than competitive games.
Some tournaments are non-FIFA (Island Games) and some names changed over time (AFF / ASEAN Championship).

I compute Elo ratings on the full history so they have time to converge (burn-in).
I train and evaluate the models on matches from 2000 onward.
I keep friendlies but give them a lower weight in the Elo update.

## 3. Non-FIFA teams
The dataset does not flag FIFA members. I use a proxy: a team that never played a World Cup qualifier since 2000 is likely not a FIFA member.

In [ ]:
non_fifa = query("""
    WITH team_matches AS (
        SELECT home_team AS team, tournament, date FROM raw_results
        UNION ALL
        SELECT away_team AS team, tournament, date FROM raw_results
    ),
    qualifier_teams AS (
        SELECT DISTINCT team
        FROM team_matches
        WHERE tournament = 'FIFA World Cup qualification'
          AND date >= '2000-01-01'
    )
    SELECT team, COUNT(*) AS n_matches, MAX(date) AS last_match
    FROM team_matches
    WHERE date >= '2000-01-01'
      AND team NOT IN (SELECT team FROM qualifier_teams)
    GROUP BY team
    ORDER BY n_matches DESC
""")

n_qualifier_teams = query("""
    SELECT COUNT(DISTINCT team) FROM (
        SELECT home_team AS team FROM raw_results
        WHERE tournament = 'FIFA World Cup qualification' AND date >= '2000-01-01'
        UNION
        SELECT away_team FROM raw_results
        WHERE tournament = 'FIFA World Cup qualification' AND date >= '2000-01-01'
    )
""").iloc[0, 0]

print(f"Teams in World Cup qualifiers since 2000: {n_qualifier_teams}")
print(f"Other teams active since 2000: {len(non_fifa)}")
non_fifa.head(30)

In [ ]:
non_fifa.tail(20)